# NYC Yellow Taxi Trip Data Analysis

## 1. Project Title & Business Objective

The objective of this project is to analyze NYC Yellow Taxi trip data for January 2025 and identify meaningful business and operational insights.

The analysis focuses on trip demand, distance, duration, passenger patterns, locations, routes, payment types, vendors, rate codes, store-and-forward records, and data quality.

The project uses Python, NumPy, Pandas, Matplotlib, and Seaborn for data cleaning, feature engineering, statistical analysis, visualization, and interpretation.

## 2. Problem Statement

The NYC Yellow Taxi dataset contains trip-level information about taxi operations in New York City during January 2025.

The analysis aims to answer:

- When is taxi demand highest and lowest?
- How does weekday demand compare with weekend demand?
- What does a typical taxi trip look like?
- Which pickup and drop-off locations are most common?
- Which routes are most frequently travelled?
- Are longer trips generally associated with longer durations?
- What passenger-count patterns exist?
- Which payment types are most common?
- How do trip characteristics vary across vendors?
- Which rate codes are most frequently used?
- What unusual or potentially problematic records exist?

## 3. Dataset Description

The dataset contains NYC Yellow Taxi trip records for January 2025 in Parquet format.

Important fields include VendorID, pickup and drop-off timestamps, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, and payment_type.

The dataset also contains fare-related columns, but this project does not analyze revenue, profit, customer spending, or driver earnings because those topics are outside the approved analytical scope.

## 4. Import Libraries

In [ ]:
# Import the required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 5. Load Dataset

In [ ]:
# Load the parquet file
df = pd.read_parquet('/content/yellow_tripdata_2025-01.parquet')
print("Dataset loaded successfully.")


## 6. Initial Data Exploration

In [ ]:
# First five records
df.head()

In [ ]:
# Last five records
df.tail()

In [ ]:
# Number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
#Column names
print("Column names:")
for column in df.columns:
    print(column)

In [ ]:
#Data types of all columns
df.dtypes

In [ ]:
# Number of unique values in each column
unique_values = df.nunique()
print("Number of unique values in each column:")
print(unique_values)

In [ ]:
#Number of missing values in each column
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

In [ ]:
#Percentage of missing values in each column
missing_percentage = (df.isnull().sum() / len(df)) * 100
print("Percentage of missing values in each column:")
print(missing_percentage.round(2))

In [ ]:
# Duplicate Rows
duplicate_rows = df.duplicated().sum()
print("Number of completely duplicated rows:", duplicate_rows)

In [ ]:
#Data types of pickup and drop-off datetime columns
print("Pickup datetime type:", df["tpep_pickup_datetime"].dtype)
print("Drop-off datetime type:", df["tpep_dropoff_datetime"].dtype)

## 7. Data Quality Assessment

In [ ]:
# Check for trips where drop-off occurs before pickup (Invalid Time)
invalid_datetime = (df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]).sum()
print("Trips where drop-off occurs before pickup:", invalid_datetime)

In [ ]:
# Check for trips where drop-off and pickup occurs same time (zero Time)
zero_duration_trips = (df["tpep_dropoff_datetime"] == df["tpep_pickup_datetime"]).sum()
print("Number of zero-duration trips:", zero_duration_trips)

In [ ]:
# Trip duration in minutes for data-quality investigation

df["trip_duration_minutes"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
print("Trip duration calculated successfully.")

In [ ]:
# Trip duration statistics
print("Minimum trip duration:", df["trip_duration_minutes"].min())
print("Maximum trip duration:", df["trip_duration_minutes"].max())
print("Mean trip duration:", df["trip_duration_minutes"].mean())
print("Median trip duration:", df["trip_duration_minutes"].median())

In [ ]:
#Count of Trips with negative duration
negative_duration = (df["trip_duration_minutes"] < 0).sum()
print("Trips with negative duration:", negative_duration)

In [ ]:
# Check for trips longer than 24 hours
long_duration_trips = (df["trip_duration_minutes"] > 24 * 60).sum()
print("Trips longer than 24 hours:", long_duration_trips)

In [ ]:
#10 longest trips

# Display the 10 longest trips

df.sort_values(by="trip_duration_minutes",ascending=False)[
    ["tpep_pickup_datetime",
     "tpep_dropoff_datetime",
     "trip_duration_minutes",
     "trip_distance",
     "passenger_count",
     "PULocationID",
     "DOLocationID"]].head(10)

In [ ]:
# Checking abnormal trip durations
print("Negative duration trips:", (df["trip_duration_minutes"] < 0).sum())
print("Zero duration trips:", (df["trip_duration_minutes"] == 0).sum())
print("Trips longer than 24 hours:", (df["trip_duration_minutes"] > 1440).sum())

In [ ]:
# Trips longer than 24 hours

long_duration_trips = df[df["trip_duration_minutes"] > 1440]
long_duration_trips[["tpep_pickup_datetime",
                     "tpep_dropoff_datetime",
                     "trip_duration_minutes",
                     "trip_distance",
                     "passenger_count",
                     "PULocationID",
                     "DOLocationID"]].sort_values("trip_duration_minutes", ascending=False)

In [ ]:
# Trips where drop-off occurs before pickup
negative_duration_trips = df[df["trip_duration_minutes"] < 0]
negative_duration_trips[["tpep_pickup_datetime",
                     "tpep_dropoff_datetime",
                     "trip_duration_minutes",
                     "trip_distance",
                     "passenger_count",
                     "PULocationID",
                     "DOLocationID"]].sort_values("trip_duration_minutes")

In [ ]:
# Zero duration trips
zero_duration_trips = df[df["trip_duration_minutes"] == 0]
zero_duration_trips[["tpep_pickup_datetime",
                     "tpep_dropoff_datetime",
                     "trip_duration_minutes",
                     "trip_distance",
                     "passenger_count",
                     "PULocationID",
                     "DOLocationID"]].head(20)

In [ ]:
#Zero-duration trips with and without distance
print("Zero-duration trips:", len(zero_duration_trips))
print("Zero-duration + zero-distance:",((zero_duration_trips["trip_distance"] == 0)).sum())
print("Zero-duration + positive-distance:",((zero_duration_trips["trip_distance"] > 0)).sum())

## 8. Data Cleaning

In [ ]:
# Create a separate cleaned dataset
clean_df = df.copy()

# Remove negative, zero, and over-24-hour trip durations
clean_df = clean_df[(clean_df["trip_duration_minutes"] > 0) &
                    (clean_df["trip_duration_minutes"] <= 1440)]

print("Original records:", len(df))
print("Records after duration cleaning:", len(clean_df))
print("Records removed:", len(df) - len(clean_df))


In [ ]:
#Trip distance values
print("Negative trip distances:", (clean_df["trip_distance"] < 0).sum())
print("Zero trip distances:", (clean_df["trip_distance"] == 0).sum())
print("Maximum trip distance:", clean_df["trip_distance"].max())

In [ ]:
#Extremely high trip distances
high_distance_trips = clean_df.sort_values("trip_distance",ascending=False).head(20)
high_distance_trips[["tpep_pickup_datetime",
                     "tpep_dropoff_datetime",
                     "trip_duration_minutes",
                     "trip_distance",
                     "passenger_count",
                     "PULocationID",
                     "DOLocationID"]]

In [ ]:
# Trip-distance percentiles
distance_percentiles = np.percentile(clean_df["trip_distance"].dropna(),[25, 50, 75, 90, 95, 99])
print("25th percentile:", distance_percentiles[0])
print("50th percentile:", distance_percentiles[1])
print("75th percentile:", distance_percentiles[2])
print("90th percentile:", distance_percentiles[3])
print("95th percentile:", distance_percentiles[4])
print("99th percentile:", distance_percentiles[5])

In [ ]:
# IQR bounds for trip distance

Q1 = np.percentile(clean_df["trip_distance"].dropna(), 25)
Q3 = np.percentile(clean_df["trip_distance"].dropna(), 75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [ ]:
# Trip-distance outliers using IQR

distance_outliers = clean_df[clean_df["trip_distance"] > upper_bound]
print("Potential distance outliers:", len(distance_outliers))
print("Percentage of clean data:",
      len(distance_outliers) / len(clean_df) * 100)

In [ ]:
#High trip-distance values
print("Trips above 10 miles:", (clean_df["trip_distance"] > 10).sum())
print("Trips above 20 miles:", (clean_df["trip_distance"] > 20).sum())
print("Trips above 50 miles:", (clean_df["trip_distance"] > 50).sum())
print("Trips above 100 miles:", (clean_df["trip_distance"] > 100).sum())
print("Trips above 1000 miles:", (clean_df["trip_distance"] > 1000).sum())

In [ ]:
#Highest-distance trips below 100 miles
clean_df[clean_df["trip_distance"] <= 100].sort_values("trip_distance",ascending=False)[["tpep_pickup_datetime",
                                                                                         "tpep_dropoff_datetime",
                                                                                         "trip_distance",
                                                                                         "trip_duration_minutes",
                                                                                          "passenger_count",
                                                                                          "PULocationID",
                                                                                          "DOLocationID"]].head(20)

In [ ]:
print("Trips above 10 miles:", (clean_df["trip_distance"] > 10).sum())
print("Trips above 20 miles:", (clean_df["trip_distance"] > 20).sum())
print("Trips above 50 miles:", (clean_df["trip_distance"] > 50).sum())
print("Trips above 100 miles:", (clean_df["trip_distance"] > 100).sum())
print("Trips above 1000 miles:", (clean_df["trip_distance"] > 1000).sum())

In [ ]:
#High-distance records
extreme_distance_trips = clean_df[clean_df["trip_distance"] > 1000].sort_values("trip_distance", ascending=False)
extreme_distance_trips[["tpep_pickup_datetime",
                        "tpep_dropoff_datetime",
                        "trip_distance",
                        "trip_duration_minutes",
                        "passenger_count",
                        "PULocationID",
                        "DOLocationID"]]

In [ ]:
#Passenger count values
print("Passenger count values:")
print(clean_df["passenger_count"].value_counts(dropna=False).sort_index())

In [ ]:
#Suspicious passenger counts
suspicious_passenger_trips = clean_df[(clean_df["passenger_count"] == 0) | (clean_df["passenger_count"] >= 7)]
print("Suspicious passenger-count records:",len(suspicious_passenger_trips))
suspicious_passenger_trips[["tpep_pickup_datetime",
                            "tpep_dropoff_datetime",
                            "trip_distance",
                            "trip_duration_minutes",
                            "passenger_count",
                            "PULocationID",
                            "DOLocationID"]].head(20)

In [ ]:
#Trips with unusually high passenger counts
high_passenger_trips = clean_df[clean_df["passenger_count"] >= 7]
high_passenger_trips[["tpep_pickup_datetime",
                      "tpep_dropoff_datetime",
                      "trip_distance",
                      "trip_duration_minutes",
                      "passenger_count",
                      "PULocationID",
                      "DOLocationID"]]

In [ ]:
#Missing values in the cleaned dataset
missing_values = clean_df.isnull().sum()
print(missing_values[missing_values > 0].sort_values(ascending=False))

In [ ]:
# Check for duplicate records
duplicate_count = clean_df.duplicated().sum()
print("Duplicate records:", duplicate_count)

In [ ]:
#Datetime column data types
print("Pickup datetime type:", clean_df["tpep_pickup_datetime"].dtype)
print("Dropoff datetime type:", clean_df["tpep_dropoff_datetime"].dtype)

In [ ]:
# pickup and drop-off location ID ranges
print("Pickup Location ID - Minimum:", clean_df["PULocationID"].min())
print("Pickup Location ID - Maximum:", clean_df["PULocationID"].max())

print("Dropoff Location ID - Minimum:", clean_df["DOLocationID"].min())
print("Dropoff Location ID - Maximum:", clean_df["DOLocationID"].max())

In [ ]:
#Unique values in store_and_fwd_flag
print(clean_df["store_and_fwd_flag"].value_counts(dropna=False))

In [ ]:
#Unique values in RatecodeID
print(clean_df["RatecodeID"].value_counts(dropna=False).sort_index())

In [ ]:
#Records with the unusual RatecodeID value 99
ratecode_99 = clean_df[clean_df["RatecodeID"] == 99]
print("RatecodeID = 99 records:", len(ratecode_99))
ratecode_99[["tpep_pickup_datetime",
             "tpep_dropoff_datetime",
             "trip_distance",
             "trip_duration_minutes",
             "passenger_count",
             "RatecodeID",
             "PULocationID",
             "DOLocationID"]].head(20)

In [ ]:
#Unique values in payment_type
print(clean_df["payment_type"].value_counts(dropna=False).sort_index())

In [ ]:
# Inspecting the payment_type value
payment_type_5 = clean_df[clean_df["payment_type"] == 5]
payment_type_5[["tpep_pickup_datetime",
                "tpep_dropoff_datetime",
                "trip_distance",
                "trip_duration_minutes",
                "passenger_count",
                "payment_type",
                "PULocationID",
                "DOLocationID"]]

In [ ]:
#Unique values in VendorID

print(clean_df["VendorID"].value_counts(dropna=False).sort_index())

In [ ]:
# Removing clearly impossible extreme trip distances
clean_df = clean_df[clean_df["trip_distance"] <= 1000]
print("Records after distance cleaning:", len(clean_df))
print("Extreme-distance records remaining:",(clean_df["trip_distance"] > 1000).sum())

In [ ]:
#Average speed for the cleaned dataset
clean_df["average_speed_mph"] = (clean_df["trip_distance"] / (clean_df["trip_duration_minutes"] / 60))
print("Minimum average speed:", clean_df["average_speed_mph"].min())
print("Maximum average speed:", clean_df["average_speed_mph"].max())

In [ ]:
#High average-speed trips
extreme_speed_trips = clean_df[clean_df["average_speed_mph"] > 100].sort_values("average_speed_mph", ascending=False)
print("Trips with speed > 100 mph:", len(extreme_speed_trips))
extreme_speed_trips[["tpep_pickup_datetime",
                      "tpep_dropoff_datetime",
                      "trip_distance",
                      "trip_duration_minutes",
                      "average_speed_mph",
                      "passenger_count",
                      "PULocationID",
                      "DOLocationID"]].head(20)

In [ ]:
#Short positive-duration trips
print("Trips <= 1 minute:",(clean_df["trip_duration_minutes"] <= 1).sum())
print("Trips <= 5 minutes:",(clean_df["trip_duration_minutes"] <= 5).sum())
print("Trips <= 10 minutes:",(clean_df["trip_duration_minutes"] <= 10).sum())

In [ ]:
#Short-duration trips with less distance
short_fast_trips = clean_df[(clean_df["trip_duration_minutes"] <= 1) & (clean_df["trip_distance"] > 1)].sort_values("trip_distance", ascending=False)
print("Trips <= 1 minute with distance > 1 mile:",len(short_fast_trips))
short_fast_trips[["tpep_pickup_datetime",
                  "tpep_dropoff_datetime",
                  "trip_distance",
                  "trip_duration_minutes",
                  "average_speed_mph",
                  "passenger_count",
                  "PULocationID",
                  "DOLocationID"]].head(20)

In [ ]:
# Remove logically impossible short-duration/high-distance trips
bad_trips = ((clean_df["trip_duration_minutes"] <= 1) & (clean_df["trip_distance"] > 1))
print("Records to remove:", bad_trips.sum())
clean_df = clean_df[~bad_trips]
print("Records after speed-related cleaning:", len(clean_df))

In [ ]:
#Average speed after cleaning
clean_df["average_speed_mph"] = (clean_df["trip_distance"] / (clean_df["trip_duration_minutes"] / 60))
print("Minimum average speed:", clean_df["average_speed_mph"].min())
print("Maximum average speed:", clean_df["average_speed_mph"].max())

In [ ]:
#Remaining very high-speed trips
fast_trips = clean_df[clean_df["average_speed_mph"] > 100].sort_values("average_speed_mph", ascending=False)
print("Trips above 100 mph:", len(fast_trips))
fast_trips[["tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "trip_duration_minutes",
            "average_speed_mph",
            "passenger_count",
            "PULocationID",
            "DOLocationID"]].head(20)

In [ ]:
# Removing trips with impossible average speed
fast_trips = clean_df["average_speed_mph"] > 100
print("Records to remove:", fast_trips.sum())
clean_df = clean_df[~fast_trips]
print("Records after speed cleaning:", len(clean_df))

In [ ]:
#Speed after cleaning
print("Minimum average speed:", clean_df["average_speed_mph"].min())
print("Maximum average speed:", clean_df["average_speed_mph"].max())
print("Trips above 100 mph:",(clean_df["average_speed_mph"] > 100).sum())

### Cleaning Decision

Statistical outliers identified using the IQR method are not automatically removed. Records are removed only when the values are clearly corrupted or logically impossible, such as extreme trip distances or calculated speeds above 100 mph.

## 9. Feature Engineering

In [ ]:
#Pickup date
clean_df["pickup_date"] = clean_df["tpep_pickup_datetime"].dt.date
print(clean_df["pickup_date"].head())

In [ ]:
#Pickup hour
clean_df["pickup_hour"] = clean_df["tpep_pickup_datetime"].dt.hour
print(clean_df["pickup_hour"].head())

In [ ]:
#Day of week
clean_df["day_of_week"] = clean_df["tpep_pickup_datetime"].dt.day_name()
print(clean_df["day_of_week"].head())

In [ ]:
#Weekend indicator
clean_df["is_weekend"] = clean_df["day_of_week"].isin(["Saturday", "Sunday"])
print(clean_df["is_weekend"].head())

In [ ]:
#Route
clean_df["route"] = (clean_df["PULocationID"].astype(str) + " → " + clean_df["DOLocationID"].astype(str))
print(clean_df["route"].head())

In [ ]:
# Distance categories
clean_df["distance_category"] = pd.cut(
    clean_df["trip_distance"],
    bins=[-np.inf, 1, 3, 10, 20, np.inf],
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"]
)
print(clean_df["distance_category"].value_counts())

In [ ]:
#Duration categories
clean_df["duration_category"] = pd.cut(clean_df["trip_duration_minutes"],
                                bins=[0, 10, 30, 60, 120, np.inf],
                                labels=["Very Short", "Short", "Medium", "Long", "Very Long"])
print(clean_df["duration_category"].value_counts())

In [ ]:
print(clean_df.columns)
print(clean_df.shape)

## 10. NumPy Statistical Analysis

In [ ]:
# trip_distance
distance = clean_df["trip_distance"].dropna().to_numpy()
print("Mean:", np.mean(distance))
print("Median:", np.median(distance))
print("Minimum:", np.min(distance))
print("Maximum:", np.max(distance))
print("Variance:", np.var(distance))
print("Standard Deviation:", np.std(distance))

print("Percentiles:")
print("25th:", np.percentile(distance, 25))
print("50th:", np.percentile(distance, 50))
print("75th:", np.percentile(distance, 75))
print("90th:", np.percentile(distance, 90))
print("95th:", np.percentile(distance, 95))
print("99th:", np.percentile(distance, 99))

In [ ]:
# trip_duration
duration = clean_df["trip_duration_minutes"].dropna().to_numpy()
print("Mean:", np.mean(duration))
print("Median:", np.median(duration))
print("Minimum:", np.min(duration))
print("Maximum:", np.max(duration))
print("Variance:", np.var(duration))
print("Standard Deviation:", np.std(duration))

print("Percentiles:")
print("25th:", np.percentile(duration, 25))
print("50th:", np.percentile(duration, 50))
print("75th:", np.percentile(duration, 75))
print("90th:", np.percentile(duration, 90))
print("95th:", np.percentile(duration, 95))
print("99th:", np.percentile(duration, 99))

In [ ]:
# passengers_count
passengers = clean_df["passenger_count"].dropna().to_numpy()
print("Mean:", np.mean(passengers))
print("Median:", np.median(passengers))
print("Minimum:", np.min(passengers))
print("Maximum:", np.max(passengers))
print("Variance:", np.var(passengers))
print("Standard Deviation:", np.std(passengers))

print("Percentiles:")
print("25th:", np.percentile(passengers, 25))
print("50th:", np.percentile(passengers, 50))
print("75th:", np.percentile(passengers, 75))
print("90th:", np.percentile(passengers, 90))
print("95th:", np.percentile(passengers, 95))
print("99th:", np.percentile(passengers, 99))

In [ ]:
# average_speed
speed = clean_df["average_speed_mph"].dropna().to_numpy()
print("Mean:", np.mean(speed))
print("Median:", np.median(speed))
print("Minimum:", np.min(speed))
print("Maximum:", np.max(speed))
print("Variance:", np.var(speed))
print("Standard Deviation:", np.std(speed))

print("Percentiles:")
print("25th:", np.percentile(speed, 25))
print("50th:", np.percentile(speed, 50))
print("75th:", np.percentile(speed, 75))
print("90th:", np.percentile(speed, 90))
print("95th:", np.percentile(speed, 95))
print("99th:", np.percentile(speed, 99))

In [ ]:
# trip_duration IQR
q1 = np.percentile(duration, 25)
q3 = np.percentile(duration, 75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

print("Lower outliers:", (duration < lower).sum())
print("Upper outliers:", (duration > upper).sum())

In [ ]:
# average_speed IQR
q1 = np.percentile(speed, 25)
q3 = np.percentile(speed, 75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

print("Lower outliers:", (speed < lower).sum())
print("Upper outliers:", (speed > upper).sum())

In [ ]:
# distance IQR
q1 = np.percentile(distance, 25)
q3 = np.percentile(distance, 75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower Bound:", lower)
print("Upper Bound:", upper)

print("Lower outliers:", (distance < lower).sum())
print("Upper outliers:", (distance > upper).sum())

## 11. Time-Based Analysis

In [ ]:
# Trips by Hour
hourly_trips = clean_df["pickup_hour"].value_counts().sort_index()
plt.figure(figsize=(10, 6))
plt.plot(hourly_trips.index, hourly_trips.values, marker="o")
plt.title("Number of Trips by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Number of Trips")
plt.xticks(range(24))
plt.grid(True)
plt.show()

In [ ]:
# Number of Trips by Day of Week
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily_trips = clean_df["day_of_week"].value_counts().reindex(day_order)
plt.figure(figsize=(10, 6))
plt.bar(daily_trips.index, daily_trips.values)
plt.title("Number of Trips by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Weekday vs Weekend
weekend_trips = clean_df["is_weekend"].value_counts()
plt.figure(figsize=(8, 6))
plt.bar(["Weekday", "Weekend"],[weekend_trips.get(False, 0), weekend_trips.get(True, 0)])
plt.title("Number of Trips: Weekday vs Weekend")
plt.xlabel("Day Type")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Trips by Pickup Hour and Day of Week
hour_day = pd.crosstab(clean_df["day_of_week"],clean_df["pickup_hour"])
plt.figure(figsize=(12, 6))
sns.heatmap(hour_day)
plt.title("Number of Trips by Pickup Hour and Day of Week")
plt.xlabel("Pickup Hour")
plt.ylabel("Day of Week")
plt.show()

In [ ]:
# Average Trip Duration by Pickup Hour
hour_duration = clean_df.groupby("pickup_hour")["trip_duration_minutes"].mean()
plt.figure(figsize=(10, 6))
plt.plot(hour_duration.index,hour_duration.values,marker="o")
plt.title("Average Trip Duration by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Average Trip Duration (minutes)")
plt.xticks(range(24))
plt.grid(True)
plt.show()

In [ ]:
# Average Trip Distance by Pickup Hour
hour_distance = clean_df.groupby("pickup_hour")["trip_distance"].mean()
plt.figure(figsize=(10, 6))
plt.plot(hour_distance.index,hour_distance.values,marker="o")
plt.title("Average Trip Distance by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Average Trip Distance (miles)")
plt.xticks(range(24))
plt.grid(True)
plt.show()

### Interpretation

Demand is concentrated in the daytime and evening, with the strongest activity around the evening peak. Early-morning hours have much lower demand. Average trip duration and average trip distance also vary by pickup hour, showing different operating patterns across the day.

## 12. Trip Distance & Duration Analysis

In [ ]:
# Distribution of Trip Distance
plt.figure(figsize=(10, 6))
sns.histplot(clean_df[clean_df["trip_distance"] <= 30]["trip_distance"],bins=50)
plt.title("Distribution of Trip Distance (Up to 30 Miles)")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Trip Duration Distribution
plt.figure(figsize=(10, 6))
sns.histplot(clean_df[clean_df["trip_duration_minutes"] <= 60]["trip_duration_minutes"],bins=50)
plt.title("Distribution of Trip Duration (Up to 60 Minutes)")
plt.xlabel("Trip Duration (minutes)")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Trip Distance vs Trip Duration
sample = clean_df.sample(10000,random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(x="trip_distance",y="trip_duration_minutes",data=sample)
plt.title("Trip Distance vs Trip Duration")
plt.xlabel("Trip Distance (miles)")
plt.ylabel("Trip Duration (minutes)")
plt.xlim(0, 30)
plt.ylim(0, 60)
plt.show()

### Interpretation

Most trips are short, while both distance and duration have long right tails. The scatter plot shows a positive relationship between distance and duration, although trips with similar distances can have different durations.

## 13. Passenger Analysis

In [ ]:
# Number of Trips by Passenger Count
passenger_counts = clean_df["passenger_count"].value_counts().sort_index()
plt.figure(figsize=(10, 6))
plt.bar(passenger_counts.index.astype(str),passenger_counts.values)
plt.title("Number of Trips by Passenger Count")
plt.xlabel("Passenger Count")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Trip Distance by Passenger Count
plt.figure(figsize=(10, 6))
sns.boxplot(x="passenger_count",y="trip_distance",data=clean_df[clean_df["passenger_count"] <= 6])
plt.title("Trip Distance by Passenger Count")
plt.xlabel("Passenger Count")
plt.ylabel("Trip Distance (miles)")
plt.ylim(0, 30)
plt.show()

In [ ]:
# Percentage of one-passenger trips
one_passenger = (clean_df["passenger_count"] == 1).sum()
passenger_total = clean_df["passenger_count"].notna().sum()
print("One-passenger trips:", one_passenger)
print("Percentage of one-passenger trips:", round(one_passenger / passenger_total * 100, 2), "%")

In [ ]:
# Passenger count by weekday and weekend
passenger_day_type = clean_df.groupby("is_weekend")["passenger_count"].mean()
print(passenger_day_type.rename({False: "Weekday", True: "Weekend"}))

### Interpretation

Single-passenger trips dominate the dataset. Trip distance and duration distributions are broadly similar across the common passenger counts, so passenger count does not appear to be a strong driver of trip distance or duration in this dataset.

## 14. Pickup & Drop-off Location Analysis

In [ ]:
# Most Common Pickup Locations
pickup_counts = clean_df["PULocationID"].value_counts()
print(pickup_counts.head(10))

In [ ]:
# Top 10 Pickup Locations
top_pickup = pickup_counts.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top_pickup.index.astype(str),top_pickup.values)
plt.title("Top 10 Pickup Locations")
plt.xlabel("Pickup Location ID")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Most Common Drop-off Locations
dropoff_counts = clean_df["DOLocationID"].value_counts()
print(dropoff_counts.head(10))

In [ ]:
# Top 10 Drop-off Locations
top_dropoff = dropoff_counts.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top_dropoff.index.astype(str),top_dropoff.values)
plt.title("Top 10 Drop-off Locations")
plt.xlabel("Drop-off Location ID")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Top 10 Routes
route_counts = clean_df["route"].value_counts()
print(route_counts.head(10))

In [ ]:
# Top 10 Routes
top_routes = route_counts.head(10)
plt.figure(figsize=(10, 6))
plt.bar(top_routes.index.astype(str),top_routes.values)
plt.title("Top 10 Taxi Routes")
plt.xlabel("Route")
plt.ylabel("Number of Trips")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Average Trip Duration for Top 10 Routes
top_route_list = route_counts.head(10).index
route_duration = clean_df[clean_df["route"].isin(top_route_list)].groupby("route")["trip_duration_minutes"].mean()
print(route_duration)

In [ ]:
# Longest average-distance route among the top 10 routes
top_route_list = route_counts.head(10).index
route_distance = clean_df[clean_df["route"].isin(top_route_list)].groupby("route")["trip_distance"].mean().sort_values(ascending=False)
print(route_distance)

### Location and Route Interpretation

A relatively small group of location IDs accounts for a large share of trips. The most frequent route is 237 → 236. Among the top 10 routes, 161 → 236 has the longest average duration, while 142 → 239 has the longest average distance.

## 15. Payment Type Analysis

In [ ]:
# Trips by Payment Type
payment_counts = clean_df["payment_type"].value_counts().sort_index()
print(payment_counts)

In [ ]:
# Payment Type Percentage
payment_percent = clean_df["payment_type"].value_counts(normalize=True).sort_index() * 100
print(payment_percent)

In [ ]:
# Trips by Payment Type
plt.figure(figsize=(10, 6))
plt.bar(payment_counts.index,payment_counts.values)
plt.title("Number of Trips by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Average Distance and Duration by Payment Type
payment_analysis = clean_df.groupby("payment_type")[["trip_distance","trip_duration_minutes"]].mean()
print(payment_analysis)

In [ ]:
# Payment Type vs Trip Distance
plt.figure(figsize=(10, 6))
sns.boxplot(x="payment_type", y="trip_distance", data=clean_df)
plt.title("Trip Distance by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Trip Distance (miles)")
plt.ylim(0, 30)
plt.show()

In [ ]:
# Payment Type vs Trip Duration
plt.figure(figsize=(10, 6))
sns.boxplot(x="payment_type", y="trip_duration_minutes", data=clean_df)
plt.title("Trip Duration by Payment Type")
plt.xlabel("Payment Type")
plt.ylabel("Trip Duration (minutes)")
plt.ylim(0, 60)
plt.show()

### Interpretation

Payment Type 1 is the most common category. The average distance and duration of Types 1 and 2 are very similar. Rare payment categories should be interpreted cautiously because some contain very few records.

## 16. Vendor Analysis

In [ ]:
# Trips by Vendor
vendor_counts = clean_df["VendorID"].value_counts().sort_index()
print(vendor_counts)

In [ ]:
# Vendor Percentage
vendor_percent = clean_df["VendorID"].value_counts(normalize=True).sort_index() * 100
print(vendor_percent)

In [ ]:
# Trips by Vendor
plt.figure(figsize=(10, 6))
plt.bar(vendor_counts.index,vendor_counts.values)
plt.title("Number of Trips by Vendor")
plt.xlabel("Vendor ID")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Average Trip Distance by Vendor
vendor_distance = clean_df.groupby("VendorID")["trip_distance"].mean()
print(vendor_distance)

In [ ]:
# Average Trip Duration by Vendor
vendor_duration = clean_df.groupby("VendorID")["trip_duration_minutes"].mean()
print(vendor_duration)

In [ ]:
# Passenger Count by Vendor
vendor_passengers = clean_df.groupby("VendorID")["passenger_count"].value_counts()
print(vendor_passengers)

In [ ]:
# Payment Type by Vendor
vendor_payment = clean_df.groupby("VendorID")["payment_type"].value_counts()
print(vendor_payment)

### Interpretation

Vendor 2 accounts for the majority of trips. Vendor-level averages can be compared descriptively, but a vendor should not be called better or worse without additional context and sufficiently large samples.

## 17. Rate Code Analysis

In [ ]:
# Trips by Rate Code
rate_counts = clean_df["RatecodeID"].value_counts().sort_index()
print(rate_counts)

In [ ]:
# Rate Code Percentage
rate_percent = clean_df["RatecodeID"].value_counts(normalize=True).sort_index() * 100
print(rate_percent)

In [ ]:
# Trips by Rate Code
plt.figure(figsize=(10, 6))
plt.bar(rate_counts.index,rate_counts.values)
plt.title("Number of Trips by Rate Code")
plt.xlabel("Rate Code")
plt.ylabel("Number of Trips")
plt.show()

In [ ]:
# Trip Distance by Rate Code
plt.figure(figsize=(10, 6))
sns.boxplot(x="RatecodeID",y="trip_distance",data=clean_df)
plt.title("Trip Distance by Rate Code")
plt.xlabel("Rate Code")
plt.ylabel("Trip Distance (miles)")
plt.ylim(0, 30)
plt.show()

In [ ]:
# Trip Duration by Rate Code
plt.figure(figsize=(10, 6))
sns.boxplot(x="RatecodeID",y="trip_duration_minutes",data=clean_df)
plt.title("Trip Duration by Rate Code")
plt.xlabel("Rate Code")
plt.ylabel("Trip Duration (minutes)")
plt.ylim(0, 60)
plt.show()

In [ ]:
# Passenger Count by Rate Code
rate_passengers = clean_df.groupby("RatecodeID")["passenger_count"].mean()
print(rate_passengers)

### Interpretation

Rate Code 1 is by far the most common category. Rare rate codes should be treated cautiously because their small sample sizes can make average distance or duration unstable.

## 18. Store & Forward Analysis

In [ ]:
# Trips by Store and Forward Flag
store_counts = clean_df["store_and_fwd_flag"].value_counts(dropna=False)
print(store_counts)

In [ ]:
# Store and Forward Percentage
store_percent = clean_df["store_and_fwd_flag"].value_counts(normalize=True) * 100
print(store_percent)

In [ ]:
# Average Trip Distance by Store and Forward Flag
store_distance = clean_df.groupby("store_and_fwd_flag")["trip_distance"].mean()
print(store_distance)

In [ ]:
# Average Trip Duration by Store and Forward Flag
store_duration = clean_df.groupby("store_and_fwd_flag")["trip_duration_minutes"].mean()
print(store_duration)

In [ ]:
# Passenger Count by Store and Forward Flag
sf_passengers = clean_df.groupby("store_and_fwd_flag")["passenger_count"].mean()
print(sf_passengers)

### Interpretation

Most non-missing store-and-forward records have the N flag. The Y category is rare, while a substantial number of records have missing values. Therefore, differences between the flags should be treated as descriptive rather than conclusive.

## 19. Correlation Analysis

In [ ]:
# CORRELATION ANALYSIS
correlation = clean_df[["trip_distance",
                        "trip_duration_minutes",
                        "passenger_count",
                        "average_speed_mph"]].corr()
print(correlation)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation,annot=True,cmap="coolwarm",fmt="0.2f")
plt.title("Correlation Between Trip Variables")
plt.show()

### Interpretation

Trip distance and average speed have the strongest positive correlation (about 0.73). Trip distance and duration have a moderate positive correlation (about 0.35). Passenger count has very weak relationships with the other variables. Correlation does not imply causation.

## 20. Advanced Analysis

In [ ]:
#Distance categories
clean_df["distance_category"] = pd.cut(clean_df["trip_distance"],
                                bins=[0, 1, 3, 10, 20, np.inf],
                                labels=["Very Short", "Short", "Medium", "Long", "Very Long"])
print(clean_df["distance_category"].value_counts())

In [ ]:
#Duration categories
clean_df["duration_category"] = pd.cut(clean_df["trip_duration_minutes"],
                                bins=[0, 10, 30, 60, 120, np.inf],
                                labels=["Very Short", "Short", "Medium", "Long", "Very Long"])
print(clean_df["duration_category"].value_counts())

In [ ]:
# Average Speed by Pickup Hour
hour_speed = clean_df.groupby("pickup_hour")["average_speed_mph"].mean()
plt.figure(figsize=(10, 6))
plt.plot(hour_speed.index,hour_speed.values,marker="o")
plt.title("Average Speed by Pickup Hour")
plt.xlabel("Pickup Hour")
plt.ylabel("Average Speed (mph)")
plt.xticks(range(24))
plt.grid(True)
plt.show()

In [ ]:
# Average Trip Duration for Top 10 Routes
top_route_list = route_counts.head(10).index
route_duration = clean_df[clean_df["route"].isin(top_route_list)].groupby("route")["trip_duration_minutes"].mean()
print(route_duration)

### Advanced Analysis Summary

The analysis also examines distance and duration categories, average speed by hour, route-level duration, and route-level distance. These analyses help identify operational patterns beyond basic descriptive statistics.

## 21. Business Questions

### 1. When is taxi demand highest?
6 PM is the busiest pickup hour, with **267,710 trips**.

### 2. When is demand lowest?
4 AM is the quietest pickup hour, with **20,005 trips**.

### 3. How does weekday demand compare with weekend demand?
Weekday demand is substantially higher than weekend demand.

### 4. What is a typical trip?
A typical cleaned trip has a median distance of **1.67 miles**, median duration of **11.7 minutes**, one passenger, and median speed of about **9.52 mph**.

### 5. What is the most common pickup location?
Location **161** is the most common pickup location.

### 6. What is the most common drop-off location?
Location **236** is the most common drop-off location.

### 7. What is the most common route?
Location **237 → 236**, with **24,819 trips**.

### 8. Do longer trips take longer?
Yes. Trip distance and duration have a moderate positive correlation of **0.348**, and average duration generally increases across distance categories.

### 9. Which hours show unusual operating patterns?
Early morning has low demand but relatively higher average speeds, while evening peak periods have high demand and lower average speeds.

### 10. What passenger pattern dominates?
One-passenger trips are overwhelmingly the most common passenger category.

### 11. Which payment type dominates?
Payment Type **1** is the most common payment category.

### 12. What vendor pattern exists?
Vendor **2** handles the majority of trips. Very small vendor samples should not be used to claim performance differences.

### 13. Which rate code dominates?
Rate Code **1** is the dominant category.

### 14. What unusual records were found?
The data contained negative/zero durations, durations above 24 hours, extreme distances, and impossible calculated speeds. Clearly corrupted records were removed.

### 15. What are the main data limitations?
The dataset covers only January 2025, uses location IDs instead of neighborhood names, contains missing categorical values, and does not support revenue, customer, or driver-level analysis.

### 16. What should be investigated next?
A longer time period, actual location names, seasonal comparisons, traffic conditions, and additional operational data would support deeper analysis.

## 22. Key Findings

- **6 PM** is the busiest pickup hour, with **267,710 trips**.
- **4 AM** is the quietest hour, with **20,005 trips**.
- **Thursday** is the busiest day, with **613,687 trips**.
- The median trip is **1.67 miles** and **11.7 minutes**.
- **One passenger** is the dominant passenger pattern.
- Location **161** is the most common pickup location and **236** is the most common drop-off location.
- The most common route is **237 → 236**, with **24,819 trips**.
- Distance and duration have a moderate positive correlation of **0.348**.
- Payment Type **1** is the most common payment category.
- Vendor **2** accounts for most trips.
- Rate Code **1** is the dominant rate category.
- Clearly corrupted records were removed while statistical outliers were retained for investigation.

### KPI Summary

The following KPIs summarize the cleaned dataset.


In [ ]:
# Final KPI Summary
print("Total Trips:", len(clean_df))
print("Average Trip Distance:", round(clean_df["trip_distance"].mean(), 2))
print("Median Trip Distance:", clean_df["trip_distance"].median())
print("Average Trip Duration:", round(clean_df["trip_duration_minutes"].mean(), 2))
print("Median Trip Duration:", clean_df["trip_duration_minutes"].median())
print("Maximum Trip Distance:", clean_df["trip_distance"].max())
print("Maximum Trip Duration:", round(clean_df["trip_duration_minutes"].max(), 2))
print("Average Speed:", round(clean_df["average_speed_mph"].mean(), 2))
print("Most Common Passenger Count:", clean_df["passenger_count"].mode()[0])
print("Busiest Pickup Hour:", clean_df["pickup_hour"].value_counts().idxmax())
print("Busiest Day:", clean_df["day_of_week"].value_counts().idxmax())
print("Most Common Payment Type:", clean_df["payment_type"].value_counts().idxmax())
print("Most Common Pickup Location:", clean_df["PULocationID"].value_counts().idxmax())
print("Most Common Drop-off Location:", clean_df["DOLocationID"].value_counts().idxmax())
print("Most Common Route:", clean_df["route"].value_counts().idxmax())


## 23. Business/Operational Insights

- Taxi capacity planning should pay particular attention to the **evening peak**, especially around 6 PM.
- Early-morning demand is much lower, which suggests a different operating pattern from daytime and evening periods.
- The concentration of trips around a limited set of pickup, drop-off, and route combinations may help identify important service corridors.
- Short trips make up a large share of the dataset, so operational analysis should account for frequent short-distance movements rather than focusing only on long trips.
- The moderate distance-duration relationship suggests that longer trips generally require more time, but distance alone does not explain trip duration.
- Vendor and payment comparisons are useful descriptively, but rare categories should not be used for strong performance conclusions.

## 24. Limitations

- The analysis uses only the **January 2025** dataset and may not represent other months or seasons.
- Pickup and drop-off values are **Location IDs**, so neighborhood-level interpretation is limited.
- Missing categorical values were retained rather than arbitrarily filled or removed.
- Some unusual but potentially valid trips remain after cleaning.
- Numeric codes for payment type, VendorID, and RatecodeID are analyzed as codes; no unsupported meanings are assigned.
- Correlation shows association, not causation.
- The analysis does not evaluate revenue, profit, customer-level behavior, or driver performance.

## 25. Final Conclusion

This project analyzed the January 2025 NYC Yellow Taxi dataset using **NumPy, Pandas, Matplotlib, and Seaborn**. The analysis covered data quality assessment, cleaning, feature engineering, statistical analysis, time-based demand, trip characteristics, passenger patterns, locations, routes, payment types, vendors, rate codes, store-and-forward records, and correlations.

The cleaned data shows that NYC Yellow Taxi trips are generally short, with a median distance of **1.67 miles** and a median duration of **11.7 minutes**. Demand is strongest in the evening, particularly at **6 PM**, and Thursday is the busiest day. One-passenger trips dominate, while a relatively small set of locations and routes accounts for a large share of activity.

The cleaning process removed clearly corrupted or logically impossible records while retaining statistical outliers that may represent valid trips. Overall, the project demonstrates how systematic data cleaning, feature engineering, statistical analysis, and visualization can turn a large trip dataset into useful operational insights.